# 01. Big Data Concepts & Distributed Computing Theory

## 🔗 Where this fits

**Builds on:** Course 02 (AIAT 112) — Unit 1, lesson 02, whose complexity analysis asked how an algorithm scales; this lesson asks the same question about the machine, and Course 05 — Unit 2, lesson 01, whose chunked loading was the first symptom.

**Used later in:** Course 05 — Unit 5, lessons 02-04, which are Dask, PySpark and RAPIDS implementations of exactly these ideas.

## Overview

- Big Data characteristics (4 Vs)
- Big Data technologies and challenges
- Distributed systems, parallel computing, and MapReduce
- Fault tolerance

These concepts underpin Dask, PySpark, and RAPIDS workflows covered in later examples.


## 1. Big Data: The Four Vs

**Volume** – Scale of data (TB, PB). Traditional single-machine tools (e.g. pandas on one laptop) cannot store or process it.

**Variety** – Mixed types: structured (tables), semi-structured (JSON, logs), unstructured (text, images, video). Different storage and processing needs.

**Velocity** – Speed of data generation and ingestion (streaming, real-time). Batch processing alone is insufficient.

**Veracity** – Quality, completeness, and trustworthiness. Big data often includes noise, missing values, and inconsistencies.

## The Story

**BEFORE**: You can work with small datasets but don't understand challenges of big data.

**AFTER**: You'll understand big data concepts: volume, velocity, variety, and strategies for handling large-scale data!

**Why this matters**: Big Data Concepts & Distributed Computing Theory is essential for building complete, professional data science solutions!

---

## 2. Big Data Technologies & Challenges

**Technologies:** Distributed storage (HDFS, S3), distributed processing (Apache Spark, Dask), message queues (Kafka), GPU acceleration (RAPIDS).

**Challenges:**
- Cost and complexity of distributed clusters
- Data locality and network bottlenecks
- Consistency, security, and governance at scale

## 3. Distributed Computing: MapReduce & Parallel Processing

**MapReduce** is a programming model for processing large datasets in parallel:
- **Map:** Apply a function to each partition; produce key–value pairs.
- **Shuffle:** Group by key.
- **Reduce:** Aggregate per key.

Spark and Dask implement MapReduce-style operations (e.g. `groupby`, `apply`) over distributed data.

## 📥 Inputs & 📤 Outputs

**Inputs:** What we use in this notebook

- Concepts: the 4 Vs, MapReduce, fault tolerance
- **Real data:** `montgomery_911_calls.csv` — the MapReduce demonstration below
  counts real 911 call categories rather than made-up letters, so the pattern is
  shown doing an actual job.

**Outputs:** What you'll see when you run the cells

- A hand-written map → group → reduce that agrees exactly with `value_counts()`

---


In [1]:
# WHAT: Implement the MapReduce pattern (map to key-value pairs, group, reduce) over real 911 call records.
# WHY: Seeing the pattern demystifies how clusters count billions of records - each phase
#      parallelizes naturally, and here we check the hand-written version against pandas.

# Conceptual MapReduce-style pattern (single-machine illustration)
from collections import defaultdict
import pandas as pd

DATA_DIR = '../../../Course 04/datasets/raw/'

# Real input: the first 50,000 rows of the Montgomery County 911 dispatch log.
# Each record is a dict, which is exactly the shape a MapReduce job receives.
calls = pd.read_csv(DATA_DIR + 'montgomery_911_calls.csv',
                    usecols=['title', 'twp'], nrows=50_000)
calls['category'] = calls['title'].str.split(':').str[0]
data = calls[['category', 'twp']].to_dict('records')
print(f"Input: {len(data):,} real 911 call records")

def map_fn(item):
    """Map: emit (category, 1) for each record."""
    category = item.get("category", "unknown")
    return (category, 1)

def reduce_fn(key, values):
    """Reduce: sum counts per key."""
    return (key, sum(values))

# MAP phase - embarrassingly parallel: every record is independent
mapped = [map_fn(d) for d in data]

# SHUFFLE phase - group by key (a real cluster moves data across the network here)
groups = defaultdict(list)
for k, v in mapped:
    groups[k].append(v)

# REDUCE phase - one worker per key
reduced = [reduce_fn(k, vals) for k, vals in groups.items()]
result = dict(sorted(reduced, key=lambda kv: -kv[1]))
print("MapReduce-style (key, count):", result)

# Check the hand-written job against pandas - they must agree exactly
expected = calls['category'].value_counts().to_dict()
print("pandas value_counts():      ", expected)
print(f"Identical: {result == expected}")
print(f"\n💡 The map phase touched {len(data):,} records independently. That is why the")
print("   pattern scales: split the input across 1,000 machines and only the shuffle")
print("   phase needs the network. The arithmetic is unchanged.")

Input: 50,000 real 911 call records
MapReduce-style (key, count): {'EMS': 24479, 'Traffic': 18099, 'Fire': 7422}
pandas value_counts():       {'EMS': 24479, 'Traffic': 18099, 'Fire': 7422}
Identical: True

💡 The map phase touched 50,000 records independently. That is why the
   pattern scales: split the input across 1,000 machines and only the shuffle
   phase needs the network. The arithmetic is unchanged.


## 4. Fault Tolerance

In distributed systems, nodes can fail. **Fault tolerance** means the system continues to work:
- **Replication:** Store data on multiple nodes; if one fails, others serve it.
- **Checkpointing:** Save intermediate results so tasks can be re-run after failure.
- **Idempotency:** Re-running a task produces the same result; safe to retry.

Spark and Dask provide fault-tolerant execution; RAPIDS typically assumes reliable GPU hardware.

## Next Steps

- **Example 02:** Dask for distributed computing
- **Example 03:** PySpark for distributed data processing
- **Example 04:** RAPIDS for GPU-accelerated workflows

## 📚 References

1. Dean, J., & Ghemawat, S. (2004). *MapReduce: Simplified Data Processing on Large Clusters*. OSDI 2004. <https://research.google/pubs/mapreduce-simplified-data-processing-on-large-clusters/>
2. Zaharia, M., Xin, R. S., Wendell, P., et al. (2016). *Apache Spark: A Unified Engine for Big Data Processing*. Communications of the ACM, 59(11), 56-65. <https://doi.org/10.1145/2934664>